<a href="https://colab.research.google.com/github/baotam27/DeepLearning/blob/main/Tuan2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Thiết lập dữ liệu giả lập dữ dằn một chút để test nhãn lạ
np.random.seed(42)
n_samples = 20

data = {
    'age': np.random.randint(20, 60, size=n_samples),
    'mileage': np.random.randint(5000, 150000, size=n_samples),
    'vehicle_type': np.random.choice(['car', 'bus', 'truck', 'motorbike'], size=n_samples),
    'level': np.random.choice(['low', 'medium', 'high'], size=n_samples),
    'price': np.random.randint(5000, 50000, size=n_samples)
}

df = pd.DataFrame(data)

# Tách biến độc lập (X) và biến phụ thuộc (y)
X = df[['age', 'mileage', 'vehicle_type', 'level']]
y = df['price']

# Thực hiện chia tập dữ liệu ngay từ đầu theo đúng Golden Rule (Split First)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(" Khởi tạo và tách dữ liệu Train/Test thành công!")

 Khởi tạo và tách dữ liệu Train/Test thành công!


Quy trình 1: Chuẩn hóa khoảng giá trị (Min-Max Scaling)

Áp dụng khi đưa các đặc trưng số về một khoảng giới hạn cố định (ví dụ $[0, 1]$).  

In [8]:
from sklearn.preprocessing import MinMaxScaler

# Lọc riêng các cột dữ liệu số
X_train_num = X_train[['age', 'mileage']]
X_test_num = X_test[['age', 'mileage']]

# Khởi tạo và chỉ Fit trên tập Train
scaler_minmax = MinMaxScaler()
scaler_minmax.fit(X_train_num)

# Biến đổi dữ liệu
X_train_minmax = scaler_minmax.transform(X_train_num)
X_test_minmax = scaler_minmax.transform(X_test_num)

print("Kết quả một phần X_train sau khi gán Min-Max:\n", X_train_minmax[:3])

Kết quả một phần X_train sau khi gán Min-Max:
 [[0.23684211 0.94718483]
 [0.97368421 0.93687676]
 [0.89473684 0.64059408]]


Quy trình 2: Chuẩn hóa phân phối chuẩn (Standardize Scaling - Z-score)

Áp dụng để đưa đặc trưng số về trung bình bằng 0 và độ lệch chuẩn bằng 1.

In [9]:
from sklearn.preprocessing import StandardScaler

# Khởi tạo và chỉ Fit trên tập Train chứa cột số
scaler_std = StandardScaler()
scaler_std.fit(X_train_num)

# Biến đổi dữ liệu cho cả hai tập
X_train_std = scaler_std.transform(X_train_num)
X_test_std = scaler_std.transform(X_test_num)

print("Kết quả một phần X_train sau khi gán Standardize:\n", X_train_std[:3])

Kết quả một phần X_train sau khi gán Standardize:
 [[-0.85875386  1.01416404]
 [ 1.43125644  0.98203081]
 [ 1.18589819  0.05843196]]


Quy trình 3: Mã hóa biến phân loại không thứ tự (One-Hot Encoding)

Sử dụng cho các biến phân loại danh nghĩa (nominal) không có thứ tự tự nhiên (ví dụ: loại xe, màu sắc). Cấu hình handle_unknown='ignore' giúp các giá trị lạ xuất hiện ở tập Test tự động chuyển thành hàng toàn số 0 thay vì làm lỗi chương trình.

In [10]:
from sklearn.preprocessing import OneHotEncoder

# Khởi tạo bộ mã hóa an toàn chống crash schema
encoder_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder_ohe.fit(X_train[['vehicle_type']])

# Biến đổi tập Train
X_train_ohe = encoder_ohe.transform(X_train[['vehicle_type']])

# Giả lập tập Test xuất hiện giá trị lạ 'bicycle' chưa từng được học ở tập Train
X_test_dummy = X_test[['vehicle_type']].copy()
X_test_dummy.iloc[0, 0] = 'bicycle'

# Biến đổi tập Test chứa giá trị lạ
X_test_ohe = encoder_ohe.transform(X_test_dummy)

print("Các phân loại học được từ Train:", encoder_ohe.categories_)
print("Dòng chứa nhãn lạ 'bicycle' tự động chuyển thành toàn số 0:\n", X_test_ohe[0])

Các phân loại học được từ Train: [array(['bus', 'car', 'motorbike', 'truck'], dtype=object)]
Dòng chứa nhãn lạ 'bicycle' tự động chuyển thành toàn số 0:
 [0. 0. 0. 0.]


Quy trình 4: Mã hóa biến phân loại có thứ tự (Ordinal Encoding)

Sử dụng cho các đặc trưng đầu vào có thứ tự rõ ràng (ví dụ: thấp, trung bình, cao). Cấu hình unknown_value=-1 để gán giá trị cố định cho các nhãn không xuất hiện trong tập Train.

In [11]:
from sklearn.preprocessing import OrdinalEncoder

# Khởi tạo và cấu hình xử lý nhãn lạ bằng giá trị -1
encoder_ordinal = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder_ordinal.fit(X_train[['level']])

# Biến đổi dữ liệu
X_train_ord = encoder_ordinal.transform(X_train[['level']])

# Giả lập tập Test có nhãn lạ 'master'
X_test_level_dummy = X_test[['level']].copy()
X_test_level_dummy.iloc[0, 0] = 'master'
X_test_ord = encoder_ordinal.transform(X_test_level_dummy)

print("Kết quả mã hóa Ordinal tập Train (3 dòng đầu):\n", X_train_ord[:3])
print("Dòng chứa nhãn lạ 'master' được xử lý an toàn thành:", X_test_ord[0])

Kết quả mã hóa Ordinal tập Train (3 dòng đầu):
 [[2.]
 [1.]
 [2.]]
Dòng chứa nhãn lạ 'master' được xử lý an toàn thành: [-1.]


Quy trình 5: Mã hóa nhãn đầu ra mục tiêu (Label Encoding)

Sử dụng riêng để chuẩn hóa nhãn kết quả $y$ (target labels) trong các bài toán phân loại.

In [12]:
from sklearn.preprocessing import LabelEncoder

# Giả lập nhãn mục tiêu y dạng chữ
y_train_label = X_train['level']
y_test_label = X_test['level']

# Khởi tạo và áp dụng duy nhất trên nhãn huấn luyện
encoder_label = LabelEncoder()
encoder_label.fit(y_train_label)

# Biến đổi nhãn kết quả
y_train_enc = encoder_label.transform(y_train_label)
y_test_enc = encoder_label.transform(y_test_label)

print("Nhãn gốc ban đầu:", list(y_train_label[:3]))
print("Nhãn mục tiêu sau khi mã hóa thành số nguyên:", y_train_enc[:3])

Nhãn gốc ban đầu: ['medium', 'low', 'medium']
Nhãn mục tiêu sau khi mã hóa thành số nguyên: [2 1 2]


Quy trình 6: Xây dựng hệ thống tự động hóa khép kín (End-to-End Pipeline)

Giải pháp tối ưu và an toàn nhất trong thực tế sử dụng thư viện Scikit-learn để gom cụm các bước tiền xử lý và mô hình hóa, tự động ngăn chặn hoàn toàn rò rỉ dữ liệu.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Khai báo phân loại nhóm cột rõ ràng
numeric_cols = ['age', 'mileage']
categorical_cols = ['vehicle_type']

# Thiết lập cấu trúc biến đổi tổng hợp
preprocess_gate = ColumnTransformer([
    ("num_gate", MinMaxScaler(), numeric_cols),
    ("cat_gate", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Đóng gói khép kín toàn bộ quy trình tiền xử lý và học máy vào Pipeline
full_pipeline = Pipeline([
    ("preprocessing", preprocess_gate),
    ("regression_model", LinearRegression())
])

# Kích hoạt vận hành hệ thống khép kín
full_pipeline.fit(X_train, y_train)

# Đánh giá kết quả cuối cùng một cách khách quan
r2_score = full_pipeline.score(X_test, y_test)
print(" Pipeline vận hành hoàn hảo!")
print("Độ chính xác của mô hình hồi quy (R2 Score):", r2_score)

 Pipeline vận hành hoàn hảo!
Độ chính xác của mô hình hồi quy (R2 Score): -0.11527972060273006
